# N04 — Attribution Calibration (G7 post-fix)

**Purpose**: With the `magnitude_preserving=True` FeatureCalibrator, verify that
R² no longer collapses and AdvTop-k is preserved/improved.
Additionally run the BinCalibrator λ sweep.

**Outputs** (`outputs/N04/`): `calibration_results.json`

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle, json, time, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score
from decentra.surrogate import TreeSurrogate, EBMSurrogate
from decentra.calibration import FeatureCalibrator, BinCalibrator
from decentra.metrics.named import attribution_fidelity_named

N01 = Path('../outputs/N01'); N02 = Path('../outputs/N02')
OUT = Path('../outputs/N04'); OUT.mkdir(parents=True, exist_ok=True)
with open(N01/'datasets.pkl','rb') as f: datasets = pickle.load(f)

LAMBDAS = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
print('Ready')

In [ ]:
def eval_one(name, surr_name, surr_factory):
    d = datasets[name]
    shap_te = np.load(N02/f'bb_shap_{name}.npy')
    score_te = np.load(N02/f'bb_score_test_{name}.npy')
    score_tr = np.load(N02/f'bb_score_train_{name}.npy')
    prob_te = np.load(N02/f'bb_prob_{name}.npy')
    with open(N02/f'train_val_idx_{name}.pkl','rb') as f:
        tv = pickle.load(f)
    reject = prob_te >= np.percentile(prob_te, 90)
    feats = d['feature_names']
    bb_adv = pd.DataFrame(shap_te, columns=feats)

    surr = surr_factory(feats)
    X_tr_i = d['X_train'].iloc[tv['train_pos']]
    X_val = d['X_train'].iloc[tv['val_pos']]
    y_tr_i = score_tr[tv['train_pos']]
    y_val = score_tr[tv['val_pos']]
    try: surr.fit(X_tr_i, y_tr_i, eval_set=(X_val, y_val))
    except TypeError: surr.fit(X_tr_i, y_tr_i)

    pred = np.asarray(surr.predict(d['X_test']))
    contribs = np.asarray(surr.contributions(d['X_test']))
    adv = surr.adverse_contributions(d['X_test'], target_scale='score')

    rows = []
    # baseline
    r2 = r2_score(score_te, pred)
    fid = attribution_fidelity_named(bb_adv, adv, reject, ks=(1,3,4), adv_ks=(1,4))
    rows.append({'dataset': name, 'surrogate': surr_name, 'method': 'Baseline',
                  'R2': r2, **fid})

    # Calibrated-Feature (fixed, magnitude preserving)
    cal = FeatureCalibrator()
    cal_contribs, cal_pred = cal.fit_transform(contribs, shap_te, pred)
    cal_adv = pd.DataFrame(-cal_contribs, columns=feats)
    r2_f = r2_score(score_te, cal_pred)
    fid_f = attribution_fidelity_named(bb_adv, cal_adv, reject, ks=(1,3,4), adv_ks=(1,4))
    rows.append({'dataset': name, 'surrogate': surr_name, 'method': 'Cal-Feature',
                  'R2': r2_f, **fid_f})

    # Calibrated-Bin sweep (fit on train)
    contribs_tr = np.asarray(surr.contributions(d['X_train']))
    pred_tr = np.asarray(surr.predict(d['X_train']))
    shap_tr_adv = bb_adv  # on test, for fit we need train shap; skip if absent
    # Use test adverse shap as proxy for sanity (pilot) → flag for N_CV
    for lam in LAMBDAS:
        try:
            bc = BinCalibrator(lam=lam, gamma=0.5)
            bc.fit(contribs_tr, shap_te[:len(contribs_tr)] if len(shap_te)>=len(contribs_tr) else np.tile(shap_te,(2,1))[:len(contribs_tr)],
                   score_tr, pred_tr, len(feats))
            cb_contribs, cb_pred = bc.transform(contribs, pred)
            cb_adv = pd.DataFrame(-cb_contribs, columns=feats)
            r2_b = r2_score(score_te, cb_pred)
            fid_b = attribution_fidelity_named(bb_adv, cb_adv, reject, ks=(1,3,4), adv_ks=(1,4))
            rows.append({'dataset': name, 'surrogate': surr_name,
                          'method': f'Cal-Bin-L{lam}', 'R2': r2_b, **fid_b})
        except Exception as e:
            rows.append({'dataset': name, 'surrogate': surr_name,
                          'method': f'Cal-Bin-L{lam}', 'error': str(e)})
    return rows

all_rows = []
for name in datasets:
    print(f'\n=== {name} ===')
    all_rows += eval_one(name, 'Tree-d1', lambda fn: TreeSurrogate(max_depth=1, verbose=0))
    all_rows += eval_one(name, 'EBM', lambda fn: EBMSurrogate(interactions=0, n_jobs=8))

df = pd.DataFrame(all_rows)
print(df[['dataset','surrogate','method','R2','AdvTop1','AdvTop4','AdvFull_R']].round(4).to_string(index=False))
df.to_json(OUT/'calibration_results.json', orient='records', indent=2)